# STEERLAB · avvio con una sola cella

**Runtime → Change runtime type → T4 GPU**, poi esegui l'unica cella qui sotto (`Ctrl+F9`). Fa tutto: clone, dipendenze, calibrazione, tunnel, modello. Alla fine stampa LINK e CHIAVE. Il repo è privato: serve `GH_TOKEN` nei Secrets di Colab (permesso `Contents: Read`).


In [ ]:
#@title STEERLAB — UNA CELLA, TUTTO AUTOMATICO (clone, GPU, calibrazione, app, link)
# Eseguire una sola volta per sessione Colab. È idempotente: rieseguirla
# riparte pulito senza rifare la calibrazione.
import os, sys, subprocess, time, re, json, shutil
from pathlib import Path

REPO   = "github.com/gio3002/steerlab-colab.git"
DIR    = "/content/steerlab-colab"
PORT   = 7860
LOCAL  = f"http://127.0.0.1:{PORT}"

def sh(cmd, **kw):
    return subprocess.run(cmd, capture_output=True, text=True, **kw)

print("=" * 72)
print("  STEERLAB · avvio automatico")
print("=" * 72, flush=True)

# ---- 1. chiave di controllo e token GitHub -------------------------------
KEY = ""
try:
    from google.colab import userdata
    KEY = (userdata.get("STEERLAB_DEPLOY_KEY") or "").strip()
except Exception:
    pass
token = ""
try:
    from google.colab import userdata
    token = (userdata.get("GH_TOKEN") or "").strip()
except Exception:
    pass

# ---- 2. codice -----------------------------------------------------------
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
if not os.path.isdir(DIR):
    r = sh(["git", "clone", url, DIR])
    if r.returncode:
        print("CLONE FALLITO (repo privato: serve GH_TOKEN nei Secrets)")
        print(r.stderr[-700:]); raise SystemExit(1)
else:
    sh(["git", "-C", DIR, "pull"])
os.chdir(DIR)
print("codice:", os.getcwd(), flush=True)

# La chiave è nel repo (control.json): solo chi ha accesso al repo ce l'ha.
try:
    ctl = json.load(open("control.json", encoding="utf-8"))
    KEY = KEY or ctl.get("deploy_key", "")
except Exception:
    ctl = {}
KEY = KEY or "__DEPLOY_KEY__"   # il demone ci inietta la chiave di sessione
assert KEY, "chiave di controllo mancante"

# ---- 3. dipendenze -------------------------------------------------------
r = sh([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
if r.returncode: print("avviso pip:", r.stderr[-300:])
if not os.path.exists("cloudflared"):
    sh(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "cloudflared"])
if os.path.exists("cloudflared"): os.chmod("cloudflared", 0o755)
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NESSUNA", flush=True)

# ---- 4. calibrazione (solo se manca o è del vecchio metodo) --------------
need = True
calp = Path("calibration/calibration.json")
if calp.exists() and Path("calibration/directions.npz").exists():
    try:
        old = json.load(open(calp, encoding="utf-8"))
        m = old.get("axes", {}).get("concision", {}).get("metrics", {}).get("method", "")
        need = "ablation" not in m
        if need: print("calibrazione vecchia -> ricalibro col nuovo equalizzatore", flush=True)
    except Exception:
        need = True
if need and not os.path.exists("equalizer.py"):
    # Il repo su GitHub non e' ancora aggiornato: calibreremmo col metodo
    # vecchio. Meglio partire subito col prior e lasciare che l'agente
    # installi il codice nuovo e ricalibri da remoto.
    print("codice vecchio sul repo: salto la calibrazione (la fara' l'agente)", flush=True)
    need = False
if need:
    print("calibrazione in corso (5-15 min)...", flush=True)
    r = subprocess.run([sys.executable, "-u", "calibrate.py"])
    print("calibrate.py exit:", r.returncode, flush=True)
else:
    print("calibrazione presente e aggiornata -> salto", flush=True)

# ---- 5. avvia app + tunnel ----------------------------------------------
sh(["pkill", "-f", "app.py --tunnel"]); time.sleep(1)
env = dict(os.environ); env["STEERLAB_DEPLOY_KEY"] = KEY
log = open("app.log", "w")
proc = subprocess.Popen([sys.executable, "-u", "app.py", "--tunnel"],
                        stdout=log, stderr=subprocess.STDOUT, env=env)
link = None
for _ in range(90):
    time.sleep(1)
    txt = open("app.log", encoding="utf-8", errors="replace").read()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
    if m: link = m.group(0); break
if not link:
    print("TUNNEL NON PARTITO — app.log:/n", txt[-1500:]); raise SystemExit(1)

# ---- 6. carica il modello ------------------------------------------------
import urllib.request
req = urllib.request.Request(LOCAL + "/api/model/load", data=b"{}",
                             headers={"Content-Type": "application/json"}, method="POST")
t0 = time.time()
try:
    with urllib.request.urlopen(req, timeout=1800) as f:
        st = json.load(f)
    print("modello:", st.get("ok"), "in %.0fs" % (time.time() - t0), flush=True)
except Exception as e:
    print("caricamento modello fallito:", e, flush=True)

# ---- 7. registra link e chiave ------------------------------------------
ctl.update({"app_url": link, "deploy_key": KEY,
            "started": time.strftime("%Y-%m-%d %H:%M:%S")})
json.dump(ctl, open("control.json", "w", encoding="utf-8"), indent=1)
print("\n" + "=" * 72)
print("  PRONTO")
print("  LINK  :", link)
print("  CHIAVE:", KEY)
print("  Apri il link nel browser. Un agente WorkBuddy puo' usare")
print("  remote_ctl.py con control.json per modificare il codice ed eseguire")
print("  esperimenti senza alcun altro intervento umano.")
print("=" * 72, flush=True)
